<center><img src="./img/pong-thumbnail.png" width="200" alt="Skills Network Logo"  /></center>
  


## Bugs & troubles:


## Dockers:

- ### Se corrompe la ruta al borrar los directorios en el host
    - Cuando levantamos los dockers a traves del archivo `Makefile` se crea el directorio `data`en el Host. Ese es el directorio donde se crean los volumenes para almacenar los datos persistentes de los volúmenes.
    - En el Makefile tenemos la opción `make clean` que borra los directorios de host para emular un borrrado accidental o intencionado.
    - Al levantar de nuevo los contenedores Docker devuelve este error:

        ```swift
        Error response from daemon: failed to copy files: failed to open target /var/lib/docker/volumes/frontend_data/_data/tsconfig.json: open /var/lib/docker/volumes/frontend_data/_data/tsconfig.json: no such file or directory
        make: *** [all] Error 1
        ````
    - En el `Makeflie`antes de lavantar los contededores revisamos si el directorio para los volumenes en el Host está creado y si no lo está, se crea.
    - Revisamos y vemos que los directorios se crean perfectamente pero Docker no los encuentra.

### Solución 1:

Parece que el problema está relacionado con la forma en que Docker maneja los volúmenes cuando eliminan los directorios del host y luego volvemos a montar los volúmenes. Esto puede causar problemas si Docker todavía tiene referencias a esos volúmenes, incluso después de haber eliminado los archivos físicamente.

***Referencia a volúmenes corruptos o huérfanos:*** Docker puede mantener referencias a volúmenes que no han sido desmontados correctamente, lo que lleva a problemas al tratar de montarlos de nuevo.

***Eliminación de volúmenes mientras están en uso:*** Si eliminamos los archivos en el host directamente, pero Docker aún está utilizando esos volúmenes, puede haber corrupción o fallos en el proceso de re-montaje.

- ### Solución sugerida antes de incluir en el Makefile:
    - Borrar la caché.

In [ ]:
docker system prune -af

### Explicación:
Limpia la caché de Docker, incluyendo contenedores parados, imágenes sin utilizar, y redes no referenciadas. Esta opción es más rápida que un reinicio y podría solucionar el problema

- La solución ```NO FUNCIONA``` sigue lanzando el mismo error cuando levantamos los contenedores.

### Solución 2 (Buena) para incluir en el Makefile:
- Reiniciar Docker

Explicamos en detalle cómo funciona Docker en relación a los volúmenes, qué puede estar causando el problema y por qué reiniciar Docker lo soluciona.:
- ### 1. ¿Qué son los volúmenes de Docker?
    Los volúmenes en Docker son áreas de almacenamiento persistente que permiten que los datos generados y utilizados por los contenedores se mantengan, incluso si los contenedores se eliminan o reinician. Son independientes del ciclo de vida de los contenedores y, generalmente, se almacenan fuera del sistema de archivos del contenedor.

    - Los volúmenes se pueden montar en rutas específicas dentro del sistema de archivos del contenedor.
    - El contenido del volumen persiste incluso si el contenedor se elimina o se recrea.
    - Los volúmenes suelen residir en una carpeta especial en el host que Docker gestiona (como /var/lib/docker/volumes en Linux o una ruta similar en macOS).
- ### 2. ¿Qué ocurre cuando eliminamos manualmente un directorio en el host una vez levantado por primera vez Dockers?
    En el proyecto, montamos directorios del host (por ejemplo, /Users/usuario/data) en contenedores Docker usando volúmenes. Cuando decidimos eliminar manualmente los directorios del host (por ejemplo, usando rm -rf), estmoa eliminando archivos que Docker espera que existan. Sin embargo:

    - **Docker aún tiene referencias a esos volúmenes en su caché** o en su base de datos interna. Aunque hayamos eliminado los archivos físicamente del host, Docker no ha sido informado de ese cambio.
    - **Docker intenta montar ese directorio eliminado** en el contenedor cuando levantamos nuevamente el contenedor, lo que provoca que se corrompa la referencia, ya que los archivos o directorios que espera ya no están.
- ### 3. ¿Por qué ocurre el error?
    El error que obtemos:

In [ ]:
Error response from daemon: failed to copy files: failed to open target /var/lib/docker/volumes/frontend_data/_data/tsconfig.json: open /var/lib/docker/volumes/frontend_data/_data/tsconfig.json: no such file or directory

Este error ocurre porque Docker intenta acceder a un archivo (en este caso `tsconfig.json`) dentro del volumen, pero ya no existe en la ruta del host que está montada. Docker está buscando ese archivo porque aún tiene metadatos y referencias internas que indican que ese archivo debería estar presente.

Esto puede ocurrir porque Docker mantiene una caché o metadatos sobre los volúmenes, y al eliminar el contenido directamente del host sin notificar a Docker, la referencia a esos archivos no se elimina correctamente en la memoria interna de Docker.


- ### 4. ¿Por qué reiniciar Docker lo soluciona?
    Cuando reiniciamos Docker, todos los procesos de Docker se detienen y se reinician, lo que incluye la limpieza de la caché interna y la actualización de los metadatos que Docker tiene sobre sus volúmenes. Básicamente:

    - Al reiniciar Docker, se refrescan las referencias que Docker tiene hacia los volúmenes.
    - Docker vuelve a verificar el estado actual de los volúmenes en el sistema de archivos del host.
    - Si los directorios o archivos ya no existen en el host, Docker ya no los tratará como válidos y evitará intentar montarlos de nuevo, eliminando la causa del error.
    
    Es decir, Docker "se da cuenta" de que los archivos que esperaba ya no están y actúa de acuerdo a esa nueva realidad, en lugar de confiar en una referencia antigua o caché que se ha quedado obsoleta.

- ### 5. ¿Qué está provocando que se corrompa la ruta?
    La ruta se corrompe porque, al eliminar los archivos manualmente desde el host mientras Docker aún los está usando o tiene referencias a ellos, Docker no tiene forma de saber que esos archivos ya no están. Como Docker sigue pensando que los archivos están presentes, y cuando intenta acceder a ellos (en nuestro caso, `tsconfig.json` dentro del volumen), el sistema devuelve un error de **"archivo no encontrado"**.

    Docker depende de la sincronización entre el estado del volumen en el host y lo que él cree que existe en ese volumen. Al eliminar manualmente esos archivos o directorios sin una intervención adecuada de Docker (como detener los contenedores y eliminar los volúmenes correctamente), introduces una discrepancia entre lo que Docker cree y lo que realmente está en el sistema de archivos del host. Esto es lo que provoca la corrupción o el fallo en la referencia.

- ### 6. ¿Cómo solucionar o mitigar este problema?
    Existen varias formas de mitigar este problema para que no tengas que reiniciar Docker cada vez:
    - **Verificar** la existencia de los archivos antes de montar los volúmenes: En el Makefile, vamos hacer una verificación condicional de que los archivos o directorios necesarios existan antes de intentar levantar los contenedores.

    - **Reiniciar Docker automáticamente** cuando se detecte un problema: Si este problema sigue ocurriendo con frecuencia, vamos a automatizar el reinicio de Docker en el Makefile. Este reinicio asegura que Docker refresque sus referencias antes de intentar montar los volúmenes de nuevo.

    - **Incluir** eliminar los volúmenes de 2 maneras: En lugar de eliminar manualmente emulando un borrado accidental o intencionado los archivos del host, vamos a usar dos comandos:
        - clean: Emula el hackeo.
        - delete: Para eliminar los volumenes de Docker controladamente con `docker volume rm`. Nos aseguramos de que Docker actualiza sus referencias internas. Esto mantendría Docker "al tanto" de que esos archivos ya no existen.
- ### 7. ¿Por qué limpiar la caché no ayuda?
    Limpiar la caché no resuelve el problema porque el problema no está en los datos en sí que Docker está almacenando temporalmente en la caché. El problema radica en las referencias internas que Docker mantiene sobre los volúmenes. Al eliminar archivos o directorios directamente desde el host sin que Docker participe en esa eliminación, las referencias se corrompen. Limpiar la caché de Docker solo afecta los datos temporales, no las referencias internas a los volúmenes.

In [ ]:
all: restart_if_needed setup
	@docker compose -f ./src/docker-compose.yml up -d --build

kill_docker:
	@./script/kill_docker.sh
	@open /Applications/Docker.app

restart_if_needed:
	@if [ ! -d "/Users/usuario/data" ]; then \
		echo "Directory /Users/usuario/data not found. Checking Docker status..."; \
		if docker ps -q > /dev/null; then \
			echo "Docker is running. Stopping Docker..."; \
			$(MAKE) kill_docker; \
		else \
			echo "Docker is not running. No need to stop Docker."; \
		fi; \
		if uname -s | grep -i darwin > /dev/null; then \
			echo "Running on macOS. Starting Docker..."; \
			open /Applications/Docker.app; \
		elif uname -s | grep -i linux > /dev/null; then \
			echo "Running on Linux. Starting Docker..."; \
			sudo systemctl start docker; \
		fi; \
		echo "Waiting for Docker to start..."; \
		sleep 10; \
		while ! docker ps > /dev/null 2>&1; do \
			echo "Waiting for Docker to be ready..."; \
			sleep 5; \
		done; \
		echo "Docker is ready."; \
	elif ! docker ps -q > /dev/null; then \
		echo "Docker is not running. Starting Docker..."; \
		if uname -s | grep -i darwin > /dev/null; then \
			echo "Running on macOS. Starting Docker..."; \
			open /Applications/Docker.app; \
		elif uname -s | grep -i linux > /dev/null; then \
			echo "Running on Linux. Starting Docker..."; \
			sudo systemctl start docker; \
		fi; \
		echo "Waiting for Docker to start..."; \
		sleep 10; \
		while ! docker ps > /dev/null 2>&1; do \
			echo "Waiting for Docker to be ready..."; \
			sleep 5; \
		done; \
		echo "Docker is ready."; \
	else \
		echo "Directory /Users/usuario/data exists. No need to restart Docker."; \
	fi


down:
	@docker compose -f ./src/docker-compose.yml down -v

clean:
	sudo rm -rf /Users/usuario/data/sqlite/*
	sudo rm -rf /Users/usuario/data/app/*
	sudo rm -rf /Users/usuario/data/php/*
	sudo rm -rf /Users/usuario/data/frontend/*
	sudo rm -rf /Users/usuario/data/blockchain/*
	sudo rm -rf /Users/usuario/data/security/*
	sudo rm -rf /Users/usuario/data
	@if docker ps -qa | grep -q .; then docker stop $$(docker ps -qa); fi
	@if docker ps -qa | grep -q .; then docker rm $$(docker ps -qa); fi
	@if docker images -qa | grep -q .; then docker rmi $$(docker images -qa); fi
	@if docker volume ls -q | grep -q .; then docker volume rm $$(docker volume ls -q); fi
	@if docker network ls --filter name=transcendence -q | grep -q .; then docker network rm transcendence; fi

setup:
	@mkdir -p /Users/usuario/data
	@mkdir -p /Users/usuario/data/sqlite
	@mkdir -p /Users/usuario/data/app
	@mkdir -p /Users/usuario/data/php
	@mkdir -p /Users/usuario/data/frontend
	@mkdir -p /Users/usuario/data/blockchain
	@mkdir -p /Users/usuario/data/security

delete:
	@docker compose -f ./src/docker-compose.yml down -v
	@if docker volume ls -qf "name=transcendence" | grep -q .; then \
		docker volume rm $$(docker volume ls -qf "name=transcendence"); \
	else \
		echo "No transcendence volumes to remove."; \
	fi

logs:
	@docker compose -f ./src/docker-compose.yml logs -f

.PHONY: all down clean setup delete logs

- ### Errores tenidos en cuenta en el Makefiel:
 - **`make`** : 
    - NO está creado el directorio de volumenes y NO corre Docker:
        - Corre Docker, crea directorio y levanta contenedores.
    - SI está creado el directorio de volumenes y NO corre Docker:
        - Corre Docker y levanta contenedores.
    - NO está creado el directorio de volumenes y SI corre Docker (borrado o hackeo):
        - Cierra Docker, Reinicia Docker y levanta contenedores para evitar perdidas de referencias.
    - Sirve para Linux y para Mac.
- **`make down`** :
    - Tumba los contenedores pero mantiene los directorios de los volumenes en el host
- **`make clean`** :
    - Elimina los directorios del host (emula borrado intencionado o por error)
- **`make delete`** :
    - Borra los volumenes de forma controlada.
- **`make logs`** :
    - Busca errores en los contenedores lavantados.
    


##  SQLite (Base de datos):

- ### No se ha creado la base de datos.
    - Listamos las tablas:
    ```shell
    docker exec -it sqlite sh
    /var/lib/sqlite # ls
    /var/lib/sqlite
    ```

No vemos nada y ejecutamos `make ps` para saber si el conteneder está en ejecución:

Vemos que el contenedor sqlite SI está en ejecución (lo cual es positivo), el siguiente paso es verificar la base de datos y asegurarnos de que el archivo db.sqlite esté disponible en el contenedor. 

In [ ]:
NAME         IMAGE                                COMMAND                  SERVICE     CREATED              STATUS                                 PORTS
app          src-backend                          "docker-entrypoint.s…"   backend     About a minute ago   Up About a minute                      0.0.0.0:3000->3000/tcp, 8080/tcp
blockchain   avaplatform/avalanchego:latest       "/avalanchego/avalan…"   avalanche   About a minute ago   Exited (1) About a minute ago          
frontend     node:18-alpine                       "docker-entrypoint.s…"   frontend    About a minute ago   Up About a minute                      0.0.0.0:8080->8080/tcp
php          php:8.1-fpm                          "docker-php-entrypoi…"   php         About a minute ago   Up About a minute                      9000/tcp
security     softwaresecurityproject/zap-stable   "sh -c '/zap/zap.sh …"   security    About a minute ago   Up About a minute (health: starting)   0.0.0.0:8081->8081/tcp
sqlite       nouchka/sqlite3                      "tail -f /dev/null"      sqlite      About a minute ago   Up About a minute           

- 1. Accede al contenedor SQLite:

Para verificar si el archivo de base de datos realmente existe en el contenedor, accede a él con:

In [ ]:
docker exec -it sqlite bash
docker exec -it sqlite sh

Esto debería abrir una shell dentro del contenedor. Luego, podemos verificar si el archivo db.sqlite está presente en el directorio /var/lib/sqlite con el siguiente comando:

In [ ]:
cd /var/lib/sqlite
ls

/var/lib/sqlite # ls
sqlite.db # Esta es la base de datos que se debería de haber creado y no ha sido así

Si el archivo NO está allí, es posible que no se haya creado correctamente o que el volumen no esté montado como esperamos; pero SI está dende debería de estar.-

- 2. Verifica el comando SQLite3:

Si el archivo sqlite.db está presente en la ubicación correcta, intentamos abrir la base de datos dentro del contenedor:

In [ ]:
sqlite3 /var/lib/sqlite/sqlite.db

sqlite3 sqlite.db

/var/lib/sqlite # sqlite3 sqlite.db
SQLite version 3.48.0 2025-01-14 11:05:00
Enter ".help" for usage hints.
sqlite> 

Ahora que hmos accedido correctamente a la base de datos SQLite dentro del contenedor, ya puedes ejecutar consultas.

In [ ]:
sqlite> .tables

Esto te mostrará una lista de todas las tablas que existen en la base de datos db.sqlite.